In [0]:
bronze_order_df = spark.read.table("retail_project.bronze.bronze_orders")

display(bronze_order_df.limit(100))

In [0]:
from pyspark.sql.functions import try_to_date, col, coalesce

silver_df = bronze_order_df.withColumn(
    'order_date',
    coalesce(
        try_to_date(col('order_date'),'yyyy-MM-dd'),
        try_to_date(col('order_date'),'dd-MM-yyyy'),
        try_to_date(col('order_date'),'yyyy/MM/dd'),
        try_to_date(col('order_date'),'dd/MM/yyyy')
    )
)
display(silver_df.limit(100))

In [0]:
silver_df.select('category').distinct().display()

In [0]:
from pyspark.sql.functions import upper, trim,when

silver_df = silver_df.withColumn(
    "category",
    upper(trim(col("category")))
)


silver_df = silver_df.withColumn(
        'category',when(col('category')=='ELECTRONIC','ELECTRONICS').
        when(col('category')=='HOME AND LIVING','HOME & LIVING').
        when(col('category')=='FASHON','FASHION').
        otherwise(col('category')))

In [0]:
silver_df.display()



In [0]:
from pyspark.sql.functions import count ,countDistinct

silver_df.select(count('order_id'),countDistinct('order_id')).display()

In [0]:
silver_df = silver_df.dropDuplicates(['order_id'])

silver_df.select(count('order_id'),countDistinct('order_id')).display()

In [0]:
from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn(
    "discount",
    when(col("discount").isNull(), "0")
    .when(col("discount") == "ten", "0.1")
    .otherwise(col("discount")),
)

In [0]:
silver_df.display()

In [0]:
from pyspark.sql.functions import regexp_replace

silver_df = silver_df.withColumn(
    "discount",
    when(col("discount").like("5%"), 
           (regexp_replace("discount", "5%", "0.05"))
    ).when(col("discount").like("10%"), 
           (regexp_replace("discount", "10%", "0.1"))
    ).otherwise(col("discount"))
)

silver_df.display()

In [0]:
silver_df = silver_df.filter(col("quantity")>=0)

In [0]:
from pyspark.sql.types import DecimalType

silver_df = silver_df.withColumn(
    "unit_price",
    col("unit_price").try_cast(DecimalType(10,2))
)
silver_df.display()

In [0]:
silver_df = silver_df.withColumn(
    "total_amount",
    col("quantity").cast("int")
    * col("unit_price")
    * (1 - col("discount").cast("double")),
)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS retail_project.silver")
silver_df.write.format("delta").mode("overwrite").saveAsTable("retail_project.silver.silver_orders")

In [0]:
customers_bronze_df = spark.table("retail_project.bronze.bronze_customers")

display(customers_bronze_df)

In [0]:
from pyspark.sql.functions import trim, col

customers_df = customers_bronze_df.select(
    trim(col("customer_id")).alias("customer_id"),
    trim(col("customer_name")).alias("customer_name"),
    trim(col("email")).alias("email"),
    trim(col("phone")).alias("phone"),
    trim(col("gender")).alias("gender"),
    trim(col("date_of_birth")).alias("date_of_birth"),
    trim(col("city")).alias("city"),
    trim(col("state")).alias("state"),
    trim(col("registration_date")).alias("registration_date"),
    trim(col("loyalty_points")).alias("loyalty_points"),
    trim(col("status")).alias("status"),
    col("ingest_timestamp")
)

display(customers_df)
     

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("customer_id") \
                    .orderBy(desc("ingest_timestamp"))

customers_df = customers_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

customers_df = customers_df.filter("row_num = 1") \
                           .drop("row_num")

customers_df.display()

In [0]:
from pyspark.sql.functions import upper, when

customers_df = customers_df.withColumn(
    "status",
    upper(col("status"))
)

customers_df = customers_df.withColumn(
    "status",
    when(col("status").isin("ACTIVE", "INACTIVE"), col("status"))
    .otherwise("UNKNOWN")
)

customers_df.display()

In [0]:
from pyspark.sql.functions import regexp_replace

customers_df = customers_df.withColumn(
    "loyalty_points",
    regexp_replace(col("loyalty_points"), "[^0-9]", "").try_cast("int")
)

In [0]:
customers_df = customers_df.withColumn(
    "email_valid",
    col("email").rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
)

customers_df.select("email", "email_valid").display()

# Example: meenadas@email.com - valid, karansingh@email - invalid

# customers_df.select("email", "email_valid").display()

customers_df = customers_df.filter(col("email_valid") == True).drop("email_valid")

In [0]:
from pyspark.sql.functions import to_date, coalesce

customers_df = (customers_df.withColumn(
    "dob_parsed",
    coalesce(
        try_to_date(col("date_of_birth"), "yyyy-MM-dd"),
        try_to_date(col("date_of_birth"), "dd-MM-yyyy"),
        try_to_date(col("date_of_birth"), "yyyy/MM/dd"),
        try_to_date(col("date_of_birth"), "dd/MM/yyyy"),
    )
).withColumn(
    "registration_parsed",
    coalesce(
        try_to_date(col("registration_date"), "yyyy-MM-dd"),
        try_to_date(col("registration_date"), "dd-MM-yyyy"),
        try_to_date(col("registration_date"), "yyyy/MM/dd"),
        try_to_date(col("registration_date"), "dd/MM/yyyy"),
    )
)
)
customers_df.display()
     

In [0]:
customers_df = customers_df.drop("date_of_birth", "registration_date") \
                           .withColumnRenamed("dob_parsed", "date_of_birth") \
                           .withColumnRenamed("registration_parsed", "registration_date")
     

In [0]:
customers_df = customers_df.withColumn(
    "gender",
    when(upper(col("gender")) == "MALE", "M")
    .when(upper(col("gender")) == "FEMALE", "F")
    .otherwise("U")
)

In [0]:
customers_df.write.format("delta").mode("overwrite").saveAsTable("retail_project.silver.silver_customers")